In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [1]:
!ls -alh ../content
!find ./ -name "*.csv"


total 12K
drwxrwxr-x 3 1000 1000 4.0K Apr 17 07:22 .
drwxrwxr-x 4 1000 1000 4.0K Apr 17 07:22 ..
drwxrwxr-x 8 1000 1000 4.0K Apr 17 07:22 TS_Timeseries
./green_.csv


In [2]:
# Re-import after reset
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

# defaltURL = "/content/drive/MyDrive/Colab Notebooks/content/TS_Timeseries/"
defaltURL = "../content/TS_Timeseries"
# All farm CSV paths
file_paths = {
    0: defaltURL+"/tom1/토마토_환경데이터_1번농가(그린CS).csv",
    1: defaltURL+"/tom4/토마토_환경데이터_4번농가(그린CS).csv",
    2: defaltURL+"/tom5/토마토_환경데이터_5번농가(그린CS).csv",
    3: defaltURL+"/tom6/토마토_환경데이터_6번농가(그린CS).csv",
}
# 사용할 센서 목록 (너가 필요하다고 판단한 센서 이름)
necessary_sensors = [
    "현재일사(W)", "내부광량",
    "온도토양", "온도급액",
    "현재EC(dS)", "EC급액", "EC배액",
    "누적량1", "저울급액", "저울배액"
]
all_dfs = []
for farm_id, path in file_paths.items():
    try:
        # CSV 읽기 (low_memory 옵션 추가해서 경고 방지)
        df = pd.read_csv(path, low_memory=False)

        # 날짜 컬럼 처리: 'date' 컬럼을 날짜형으로 변환 및 정렬
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df = df.sort_values('date').reset_index(drop=True)

        # 필요한 센서만 선택: 실제 데이터에 있는 컬럼과 교집합
        cols_to_use = [col for col in necessary_sensors if col in df.columns]
        if not cols_to_use:
            print(f"Farm {farm_id} skipped: No necessary sensor columns found.")
            continue
        df = df[cols_to_use].copy()

        # 문자열 또는 비숫자 데이터를 숫자형으로 변환 (변환 불가하면 NaN)
        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        # 객체형(dtype) 추론
        df = df.infer_objects()

        # 결측치 보간: 선형 보간법 적용 (양방향)
        df = df.interpolate(method='linear', limit_direction='both')

        # 만약 남은 NaN이 있다면 0으로 대체 (필요에 따라 다른 대체값 사용 가능)
        df = df.fillna(0)

        # StandardScaler를 이용한 정규화
        scaler = StandardScaler()
        df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

        # 농가 식별용 farm_id 컬럼 추가
        df_scaled["farm_id"] = farm_id

        all_dfs.append(df_scaled)

    except Exception as e:
        print(f"Error in farm {farm_id}: {e}")

# 모든 농가 데이터 통합
if all_dfs:
    df_all_selected = pd.concat(all_dfs, ignore_index=True)
    print("정규화 및 센서 선택 후 통합 데이터프레임 미리보기 (상위 5개):")
    print(df_all_selected.head())

    print("\n[농가별 샘플 수]")
    print(df_all_selected['farm_id'].value_counts())
else:
    print("모든 농가 데이터 처리 실패!")


정규화 및 센서 선택 후 통합 데이터프레임 미리보기 (상위 5개):
    현재일사(W)      내부광량      온도토양      온도급액  현재EC(dS)      EC급액      EC배액  \
0 -0.577172  1.401695  0.106297  0.103445  1.609545 -0.010014  0.803523   
1 -0.577172  1.401695  0.106297  0.103445  1.270037 -0.010014  0.803523   
2 -0.577172  1.401695  0.106297  0.103445  1.609545 -0.010014  0.803523   
3 -0.577172  1.401695  0.106297  0.103445  1.609545 -0.010014  0.803523   
4 -0.577172  1.401695  0.106297  0.103445  1.609545 -0.010014  0.803523   

       누적량1      저울급액      저울배액  farm_id  
0  2.492593  0.380625  0.179108        0  
1  2.492593  0.380625  0.179108        0  
2  2.492593  0.380625  0.179108        0  
3  2.492593  0.380625  0.179108        0  
4  2.492593  0.380625  0.179108        0  

[농가별 샘플 수]
farm_id
0    201669
3    162144
1    152658
2    130125
Name: count, dtype: int64


In [3]:
# 모든 농가 데이터 통합
df_all_scaled = pd.concat(all_dfs, ignore_index=True)

# 결과 미리보기
# 정규화된 모든 농가 데이터 미리보기 (상위 5개)
print(df_all_scaled.head())

# 각 농가별 데이터 개수 요약
print("\n[농가별 샘플 수]")
print(df_all_scaled['farm_id'].value_counts())

    현재일사(W)      내부광량      온도토양      온도급액  현재EC(dS)      EC급액      EC배액  \
0 -0.577172  1.401695  0.106297  0.103445  1.609545 -0.010014  0.803523   
1 -0.577172  1.401695  0.106297  0.103445  1.270037 -0.010014  0.803523   
2 -0.577172  1.401695  0.106297  0.103445  1.609545 -0.010014  0.803523   
3 -0.577172  1.401695  0.106297  0.103445  1.609545 -0.010014  0.803523   
4 -0.577172  1.401695  0.106297  0.103445  1.609545 -0.010014  0.803523   

       누적량1      저울급액      저울배액  farm_id  
0  2.492593  0.380625  0.179108        0  
1  2.492593  0.380625  0.179108        0  
2  2.492593  0.380625  0.179108        0  
3  2.492593  0.380625  0.179108        0  
4  2.492593  0.380625  0.179108        0  

[농가별 샘플 수]
farm_id
0    201669
3    162144
1    152658
2    130125
Name: count, dtype: int64


In [4]:
# 사용할 센서 목록 (모델 학습에 필요한 센서)
necessary_sensors = [
    "현재일사(W)", "내부광량",
    "온도토양", "온도급액",
    "현재EC(dS)", "EC급액", "EC배액",
    "누적량1", "저울급액", "저울배액"
]

# 기존에 처리한 전체 통합 데이터프레임(df_all_scaled)에서 필요한 센서만 선택
# (만약 df_all_scaled가 아니라 각 농가별 처리가 끝난 후의 DataFrame이라면, 그 단계에서 미리 필터링하면 좋음)
df_filtered = df_all_scaled[necessary_sensors + ["farm_id"]]

# 결과 미리보기
print("필요 센서만 남긴 데이터프레임 (상위 5개):")
print(df_filtered.head())

# 각 농가별 샘플 수 확인
print("\n[필요 센서만 남긴 후 농가별 샘플 수]")
print(df_filtered['farm_id'].value_counts())


필요 센서만 남긴 데이터프레임 (상위 5개):
    현재일사(W)      내부광량      온도토양      온도급액  현재EC(dS)      EC급액      EC배액  \
0 -0.577172  1.401695  0.106297  0.103445  1.609545 -0.010014  0.803523   
1 -0.577172  1.401695  0.106297  0.103445  1.270037 -0.010014  0.803523   
2 -0.577172  1.401695  0.106297  0.103445  1.609545 -0.010014  0.803523   
3 -0.577172  1.401695  0.106297  0.103445  1.609545 -0.010014  0.803523   
4 -0.577172  1.401695  0.106297  0.103445  1.609545 -0.010014  0.803523   

       누적량1      저울급액      저울배액  farm_id  
0  2.492593  0.380625  0.179108        0  
1  2.492593  0.380625  0.179108        0  
2  2.492593  0.380625  0.179108        0  
3  2.492593  0.380625  0.179108        0  
4  2.492593  0.380625  0.179108        0  

[필요 센서만 남긴 후 농가별 샘플 수]
farm_id
0    201669
3    162144
1    152658
2    130125
Name: count, dtype: int64


이제 센서 선택과 전처리를 완료했으니, 이제 남은 단계는 모델 학습을 위한 데이터셋 구축과 Transformer 기반 예측 모델 설계 및 학습이야. 단계별로 정리해보자:

# 1. 시계열 슬라이딩 윈도우 시퀀스 생성
목표:

시간을 기반으로 고정 길이의 윈도우를 만들어서 모델에 입력할 수 있는 시계열 샘플을 생성

예를 들어, 과거 48타임스텝을 입력으로 주고, 이후 12타임스텝의 값을 예측하도록 구성할 수 있어.

실행 단계:

df_filtered (필요 센서만 남긴 데이터프레임)을 시간 순서대로 정렬한 후, 윈도잉 방식으로 슬라이딩 윈도우 생성

각 샘플은 [입력, 출력, farm_id]를 포함해야 해

In [5]:
def create_sliding_windows(df, feature_cols, target_col, input_len=48, output_len=12):
    X, y, farm_ids = [], [], []
    total_len = input_len + output_len
    for i in range(len(df) - total_len):
        window = df.iloc[i : i + total_len]
        # 입력: 처음 input_len 타임스텝의 feature 값
        X.append(window[feature_cols].iloc[:input_len].values)
        # 출력: 뒤 output_len 타임스텝의 target 값 (예: 현재EC(dS))
        y.append(window[target_col].iloc[input_len:].values)
        # farm_id: 해당 윈도우의 시작 farm_id (모든 타임스텝이 동일할 것으로 가정)
        farm_ids.append(window["farm_id"].iloc[0])
    return np.array(X), np.array(y), np.array(farm_ids)

# 사용할 센서 목록 (필요 센서만 선정)
necessary_sensors = [
    "현재일사(W)", "내부광량", "온도토양", "온도급액",
    "현재EC(dS)", "EC급액", "EC배액", "누적량1", "저울급액", "저울배액"
]
target_sensor = "현재EC(dS)"  # 예측 타겟 선택

# 데이터 프레임 df_filtered는 앞서 생성한 필터링된 데이터임
# 시간 순서대로 정렬이 이미 되어 있다고 가정 (필요하면 df_filtered = df_filtered.sort_values('date') 등으로 처리)
X_seq, y_seq, farm_ids_seq = create_sliding_windows(df_filtered, necessary_sensors, target_sensor, input_len=48, output_len=12)
print("입력 시퀀스 shape:", X_seq.shape)
print("출력 시퀀스 shape:", y_seq.shape)
print("farm_id shape:", farm_ids_seq.shape)


입력 시퀀스 shape: (646536, 48, 10)
출력 시퀀스 shape: (646536, 12)
farm_id shape: (646536,)


# 2. PyTorch Dataset 및 DataLoader 구성
목표:

생성한 시퀀스 데이터를 PyTorch Dataset 클래스로 래핑해서, 학습 시 mini-batch 로 처리할 수 있도록 함.

예제 코드:

In [8]:
import typing_extensions
print("🔍 로딩된 경로:", typing_extensions.__file__)

🔍 로딩된 경로: /opt/conda/lib/python3.10/site-packages/typing_extensions.py


In [9]:
import torch
from torch.utils.data import Dataset, DataLoader

class SensorTimeSeriesDataset(Dataset):
    def __init__(self, X, y, farm_ids):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.farm_ids = torch.tensor(farm_ids, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.farm_ids[idx]

# Dataset 인스턴스 생성
dataset = SensorTimeSeriesDataset(X_seq, y_seq, farm_ids_seq)
# DataLoader 생성 (예: batch_size = 64)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)


# 3. Transformer 기반 예측 모델 설계 (Farm ID Embedding 포함)
목표:

시계열 데이터를 입력받아 예측하는 Transformer 모델을 구축하고, farm_id를 임베딩하여 농가 특성을 반영

예제 코드:

In [10]:
import torch
import torch.nn as nn

class FarmAwareTransformer(nn.Module):
    def __init__(self, input_dim, model_dim=64, num_heads=4,
                 num_layers=3, output_len=12,
                 num_farms=4, farm_embed_dim=4):
        super().__init__()
        self.farm_embed = nn.Embedding(num_farms, farm_embed_dim)
        self.input_proj = nn.Linear(input_dim + farm_embed_dim, model_dim)

        # batch_first=True로 설정
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            enable_nested_tensor=True
        )

        self.output_proj = nn.Linear(model_dim, output_len)

    def forward(self, x, farm_ids):
        # x: [batch, seq_len, input_dim]
        farm_embed = self.farm_embed(farm_ids)                   # [batch, farm_embed_dim]
        farm_embed_exp = farm_embed.unsqueeze(1).expand(         # [batch, seq_len, farm_embed_dim]
            -1, x.size(1), -1
        )
        x = torch.cat([x, farm_embed_exp], dim=-1)               # [batch, seq_len, input_dim+farm_embed_dim]
        x = self.input_proj(x)                                  # [batch, seq_len, model_dim]
        # batch_first=True이므로 permute 불필요
        # x = x.permute(1, 0, 2)
        encoded = self.encoder(x)                               # [batch, seq_len, model_dim]
        final_repr = encoded[:, -1, :]                          # [batch, model_dim]
        out = self.output_proj(final_repr)                      # [batch, output_len]
        return out

# 인스턴스화 예시
model = FarmAwareTransformer(input_dim=10)
print(model)


FarmAwareTransformer(
  (farm_embed): Embedding(4, 4)
  (input_proj): Linear(in_features=14, out_features=64, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (output_proj): Linear(in_features=64, out_features=12, bias=True)
)


# 4. 학습 루프 구성
목표:

DataLoader에서 배치별로 데이터를 받아서 모델을 학습시키는 루프 작성

손실 함수(MSELoss) 및 옵티마이저(Adam 등) 적용

예제 코드:

In [ ]:
import torch
print(torch.cuda.is_available())  # True여야 GPU 사용 가능


In [12]:
import torch

model.load_state_dict(torch.load("/content/drive/MyDrive/Colab Notebooks/Tomato_Timeseries/greenscs_trainedCSAll.pt"))
model.train()  # 계속 학습 모드로 설정

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/Tomato_Timeseries/greenscs_trainedCSAll.pt'

In [ ]:
import torch.optim as optim

# 손실함수 및 옵티마이저 정의
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 10  # 예시

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch_X, batch_y, batch_farm in dataloader:
        optimizer.zero_grad()
        # 모델 예측; batch_X: [batch, seq_len, input_dim]
        pred = model(batch_X, batch_farm)  # [batch, output_len]
        loss = criterion(pred, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch_X.size(0)
    avg_loss = total_loss / len(dataset)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.6f}")


In [ ]:

# 학습이 끝나면 모델을 저장
torch.save(model.state_dict(), "/content/drive/MyDrive/Colab Notebooks/Tomato_Timeseries/greenscs_trainedCSAll.pt")

# 5. 평가 및 추가 개선
학습이 완료되면:

Validation 데이터셋을 통해 모델 성능을 평가하고, 필요에 따라 하이퍼파라미터, 모델 아키텍처, 데이터 전처리 과정을 조정

추가적으로, 새로운 농가 데이터에 대해 모델을 Fine-Tuning하는 방법 등을 고려할 수 있음

요약
슬라이딩 윈도우 시퀀스 생성 → 시간 기반 시계열 입력/출력 샘플 구축

PyTorch Dataset과 DataLoader → 배치 학습 준비

Transformer 모델 (FarmAwareTransformer) → 센서 데이터 + farm_id 임베딩 활용

학습 루프 구현 → 모델 학습 및 평가

이 흐름대로 하나씩 진행하면 모델 학습을 위한 전체 파이프라인이 완성될 거야.

In [ ]:
import torch

# 모델 평가 모드로 전환
model.eval()

# 평가 중에는 그래디언트 계산을 하지 않도록 함
with torch.no_grad():
    # 예시: DataLoader에서 배치 하나를 가져옴 (test_dataloader가 있거나, 같은 dataloader를 사용해도 됨)
    for batch_X, batch_y, batch_farm in dataloader:
        predictions = model(batch_X, batch_farm)
        # 예측 결과와 실제 값을 출력 (예: 첫 5개 샘플)
        print("예측 결과:")
        print(predictions[:5])
        print("실제 값:")
        print(batch_y[:5])
        break  # 첫 배치만 테스트


In [ ]:
import torch
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 예시: DataLoader에서 첫 번째 배치를 가져와서 테스트
model.eval()
with torch.no_grad():
    for batch_X, batch_y, batch_farm in dataloader:
        predictions = model(batch_X, batch_farm)  # 모델 예측 결과, shape: [batch, output_len]
        actual = batch_y  # 실제 값, shape: [batch, output_len]
        break  # 첫 배치만 사용

# GPU 텐서일 경우, CPU로 이동 후 numpy 배열로 변환
pred_np = predictions.cpu().numpy() if torch.is_tensor(predictions) else predictions
actual_np = actual.cpu().numpy() if torch.is_tensor(actual) else actual

# MSE와 MAE 계산 (각 배치의 예측 성능)
mse = mean_squared_error(actual_np, pred_np)
mae = mean_absolute_error(actual_np, pred_np)

print(f"MSE: {mse:.6f}")
print(f"MAE: {mae:.6f}")


In [ ]:
import matplotlib.pyplot as plt
import torch

# 모델을 평가 모드로 전환
model.eval()

with torch.no_grad():
    # 첫 번째 배치를 가져오기
    for batch_X, batch_y, batch_farm in dataloader:
        predictions = model(batch_X, batch_farm)
        actual = batch_y  # 실제 값 할당
        break  # 첫 배치만 사용

# GPU 텐서일 경우 CPU로 이동하고 numpy array로 변환
pred_np = predictions.cpu().numpy() if torch.is_tensor(predictions) else predictions
actual_np = actual.cpu().numpy() if torch.is_tensor(actual) else actual

# 예를 들어, 첫 5개 샘플에 대해 예측 결과와 실제 값을 그래프로 비교
num_samples_to_plot = 5
for i in range(num_samples_to_plot):
    plt.figure(figsize=(10, 4))
    plt.plot(actual_np[i], 'o-', label='Actual')
    plt.plot(pred_np[i], 'x--', label='Predicted')
    plt.xlabel("Time Step")
    plt.ylabel("Normalized Value")
    plt.title(f"Sample {i} Prediction vs Actual")
    plt.legend()
    plt.grid(True)
    plt.show()
